
#### Requisitos para o arquivo torneios.csv

Objetivo: realizar um processo de ETL e disponibilizar os dados em uma camada apropriada

Ler -> Transformar -> Escrever (ETL)

1. Ler o arquivo csv do Data Lake
2. Renomear as colunas para português utilizando o **snake case**
3. Definir o esquema de dados correto
4. Precisa ter uma coluna com a data em que o arquivo foi consumido
5. Salvar os dados em formato **parquet** na camada adequada
6. Bônus: Criar uma coluna com a palavra 'Brasil'


#### 1. Lendo o arquivo CSV

In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
%run "../Modulo 4/Variaveis"

In [0]:
%run "../Modulo 4/funcoes"

In [0]:
caminho_arquivo_torneios = f"{caminho_bronze}torneios.csv"


In [0]:
display(dbutils.fs.ls(caminho_bronze))

In [0]:
torneios_df = (spark.read
               .option("encoding", "UTF-16")
               .option("header", "true")
               .option("sep", ",")
               .option("inferSchema", "true")
               .csv(caminho_arquivo_torneios))

In [0]:
display(torneios_df)

In [0]:
torneios_df.printSchema()

In [0]:
torneios_df_renomeado = torneios_df \
    .withColumnRenamed("league_id", "id_torneio") \
    .withColumnRenamed("league_name", "nm_torneio") \
    .withColumnRenamed("country_name", "nm_pais") \
    .withColumnRenamed("country_code", "cd_pais") \
    .withColumnRenamed("season_year", "ano_torneio") \
    .withColumnRenamed("season_start", "dt_inicio") \
    .withColumnRenamed("season_end", "dt_fim") \
    .withColumnRenamed("season_current", "e_atual")

In [0]:
torneios_df_renomeado.display()

In [0]:
from pyspark.sql.types import BooleanType

In [0]:
torneios_df_retipado = torneios_df_renomeado.withColumn(
    "e_atual", torneios_df_renomeado.e_atual.cast(BooleanType())
)

In [0]:
torneios_df_retipado.printSchema()

In [0]:
torneios_df_retipado.display()

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
torneios_df_data = cria_coluna_data(torneios_df_retipado)

In [0]:
display(torneios_df_data)

In [0]:
torneios_df_data.printSchema()

In [0]:
torneios_df_final = torneios_df_data.withColumn("mm_traduzido", lit("Brasil"))

In [0]:
display(torneios_df_final)

In [0]:
torneios_df_data.write.parquet(f"{caminho_silver}torneios/", mode="overwrite")